In [1]:
import torch
import tqdm
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
import os
from sklearn.metrics import precision_recall_fscore_support
from torch.optim.lr_scheduler import ReduceLROnPlateau, CosineAnnealingLR
import ssl
ssl._create_default_https_context = ssl._create_unverified_context

import torch.nn.functional as F

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [2]:
!nvidia-smi

Thu Oct 31 01:22:18 2024       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.183.01             Driver Version: 535.183.01   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  NVIDIA GeForce RTX 3090        Off | 00000000:01:00.0 Off |                  N/A |
|  0%   44C    P5              58W / 350W |  13164MiB / 24576MiB |      0%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+--

In [3]:
class ResNet6(nn.Module):
    def __init__(self, num_classes=10, channels=3):
        super(ResNet6, self).__init__()
        self.conv1 = nn.Conv2d(in_channels=channels, out_channels=64, kernel_size=3, stride=1, padding=1)
        self.bn1 = nn.BatchNorm2d(64)
        self.conv2 = nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, stride=1, padding=1)
        self.bn2 = nn.BatchNorm2d(128)
        self.conv3 = nn.Conv2d(in_channels=128, out_channels=256, kernel_size=3, stride=1, padding=1)
        self.bn3 = nn.BatchNorm2d(256)
        self.conv4 = nn.Conv2d(in_channels=256, out_channels=512, kernel_size=3, stride=1, padding=1)
        self.bn4 = nn.BatchNorm2d(512)
        self.fc = nn.Linear(512, num_classes)

    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.relu(self.bn2(self.conv2(x)))
        x = F.max_pool2d(x, 2)
        x = F.relu(self.bn3(self.conv3(x)))
        x = F.relu(self.bn4(self.conv4(x)))
        x = F.max_pool2d(x, 2)
        x = F.avg_pool2d(x, x.size()[2:])
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x
    
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image

# Modify the first layer to handle 5 channels
class Conv2dAdapter(nn.Module):
    def __init__(self):
        super(Conv2dAdapter, self).__init__()
        # Replace the first conv2d to handle 5 input channels (instead of 3 for RGB)
        self.conv2d = nn.Conv2d(in_channels=5, out_channels=96, kernel_size=(4, 4), stride=(4, 4))

    def forward(self, x):
        # Input shape: [batch_size, 5, 128, 128]
        x = self.conv2d(x)  # Apply 2D convolution, output shape: [batch_size, 96, H, W]
        return x


class SwinV2t(nn.Module):
    def __init__(self, num_classes=10):
        super(SwinV2t, self).__init__()

        self.SwinV2t = models.swin_v2_t(weights="IMAGENET1K_V1")
        # Replace the initial Conv2d layer in Swin with your modified Conv2d layer
        self.SwinV2t.features[0][0] = Conv2dAdapter()
        self.SwinV2t.head = nn.Linear(in_features=768, out_features=num_classes, bias=True)
        
    def forward(self, x):
        out = self.SwinV2t(x)
        
        return out

In [4]:
sentinel_model = SwinV2t(11255)
sentinel_model.load_state_dict(torch.load("../models/SwinV2t-Sentinel2A-bestF1-seed_1_opt_AdamW_lr_0.001_batch_32_weight1.pth"))
sentinel_model.SwinV2t.head = nn.Identity()

In [5]:
# Define modified ResNet18 model to accept 4-channel input
class MultiModalEnsembleC(nn.Module):
    def __init__(self, num_classes):
        super(MultiModalEnsembleC, self).__init__()
        
        self.landsat_model = ResNet6(channels=6, num_classes=num_classes)
        self.landsat_model.load_state_dict(torch.load("ResNet6-LandsatCubes-bestF1-seed_123_opt_AdamW_lr_0.0001_batch_32_weight1.pth"))
        self.landsat_model.fc = nn.Identity()
        
        self.bioclim_model = ResNet6(channels=4, num_classes=num_classes)
        self.bioclim_model.load_state_dict(torch.load("ResNet6-BioclimCubes-bestF1-seed_123_opt_AdamW_lr_0.0001_batch_32_weight10.pth"))
        self.bioclim_model.fc = nn.Identity()
        
        self.sentinel_model = ResNet6(channels=4, num_classes=num_classes)
        self.sentinel_model.load_state_dict(torch.load("ResNet6-Sentinel2-bestF1-seed_66_opt_AdamW_lr_0.0001_batch_32_weight1.pth"))
        self.sentinel_model.fc = nn.Identity()
                
        self.fc1 = nn.Linear(512+512+512, num_classes)

    def forward(self, x, y, z):
        
        x = self.landsat_model(x)
        y = self.bioclim_model(y)
        z = self.sentinel_model(z)
        xyz = torch.cat((x, y, z), dim=1)
        
        out = self.fc1(xyz)
                
        return out

In [6]:
# Check that MPS is available
device = torch.device("cpu")

if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("DEVICE = MPS")

if torch.cuda.is_available():
    device = torch.device("cuda:1")
    print("DEVICE = CUDA")

DEVICE = CUDA


In [7]:
def set_seed(seed):
    # Set seed for Python's built-in random number generator
    torch.manual_seed(seed)
    # Set seed for numpy
    np.random.seed(seed)
    # Set seed for CUDA if available
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        # Set cuDNN's random number generator seed for deterministic behavior
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

# Usage example:
set_seed(69)

In [8]:
def construct_patch_path(data_path, survey_id):
    """Construct the patch file path based on plot_id as './CD/AB/XXXXABCD.jpeg'"""
    path = data_path
    for d in (str(survey_id)[-2:], str(survey_id)[-4:-2]):
        path = os.path.join(path, d)

    path = os.path.join(path, f"{survey_id}.jpeg")

    return path

In [9]:
# EarlyStopping class to stop training when validation performance plateaus
class EarlyStopping:
    def __init__(self, patience=5, verbose=False, delta=0, path='best_model.pth'):
        """
        Args:
            patience (int): How long to wait after last time validation loss improved.
                            Default: 5
            verbose (bool): If True, prints a message for each validation loss improvement. 
                            Default: False
            delta (float): Minimum change in the monitored quantity to qualify as an improvement.
                            Default: 0
            path (str): Path to save the best model. Default: 'best_model.pth'
        """
        self.patience = patience
        self.verbose = verbose
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.val_loss_min = np.inf
        self.delta = delta
        self.path = path

    def __call__(self, val_loss, model):
        score = -val_loss

        if self.best_score is None:
            self.best_score = score
            self.save_checkpoint(val_loss, model)
        elif score < self.best_score + self.delta:
            self.counter += 1
            if self.verbose:
                print(f"EarlyStopping counter: {self.counter} out of {self.patience}")
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            self.save_checkpoint(val_loss, model)
            self.counter = 0

    def save_checkpoint(self, val_loss, model):
        '''Saves model when validation loss decreases.'''
        if self.verbose:
            print(f"Validation loss decreased ({self.val_loss_min:.6f} --> {val_loss:.6f}).  Saving model ...")
        torch.save(model.state_dict(), self.path)
        self.val_loss_min = val_loss

In [10]:
import torch
import numpy as np
import wandb
from sklearn.metrics import precision_recall_fscore_support, roc_auc_score
from torch.utils.data import DataLoader
from sklearn.model_selection import KFold
from sklearn.model_selection import train_test_split


def create_optimizer(params, model):
    """
    Creates and returns the optimizer based on the parameter configuration.
    Args:
        params (dict): Contains 'optimizer', 'lr', and other hyperparameters.
        model (nn.Module): The model to optimize.
    Returns:
        optimizer (torch.optim.Optimizer): The initialized optimizer.
    """
    if params['optimizer'] == 'Adam':
        return optim.Adam(model.parameters(), lr=params['lr'])
    if params['optimizer'] == 'AdamW':
        return optim.AdamW(model.parameters(), lr=params['lr'])
    elif params['optimizer'] == 'SGD':
        return optim.SGD(model.parameters(), lr=params['lr'], momentum=0.9)
    elif params['optimizer'] == 'RMSprop':
        return optim.RMSprop(model.parameters(), lr=params['lr'], momentum=0.9)
    else:
        raise ValueError(f"Unsupported optimizer: {params['optimizer']}")

# Custom training function with early stopping, model saving, and optimizer selection
def train_and_validate(model, train_loader, val_loader, config, early_stopping):
    best_loss = np.inf
    best_f1 = 0
    best_model_path = early_stopping.path
    
    # Create optimizer based on the config
    optimizer = create_optimizer(config, model)
    
    # Scheduler can be used for learning rate adjustments
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=1, factor=0.9)

    for epoch in range(config['num_epochs']):
        model.train()
        train_loss = 0.0
        
        for batch_idx, (landsat, bioclim, sentinel, targets, _) in enumerate(train_loader):
            landsat = landsat.to(config['device'])
            bioclim = bioclim.to(config['device'])
            sentinel = sentinel.to(config['device'])
            targets = targets.to(config['device'])

            optimizer.zero_grad()
            outputs = model(landsat, bioclim, sentinel)
            pos_weight = targets * config['positive_weight_factor']
            criterion = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
            
        train_loss /= len(train_loader)
        print(f"Epoch {epoch+1}/{config['num_epochs']}, Train Loss: {train_loss}")
        wandb.log({"train/loss": train_loss, "train/lr": optimizer.param_groups[0]['lr']})

        # Validation
        val_loss, val_f1, val_precision, val_recall = validate(model, val_loader, config)
        wandb.log({"val/loss": val_loss, "val/sF1@25": val_f1, "val/sPrecision@25": val_precision, "val/sRecall@25": val_recall})

        # Early stopping checks
        early_stopping(val_loss, model)
        if early_stopping.early_stop:
            print("Early stopping triggered!")
            break

        scheduler.step(val_loss)
        print("Scheduler:", scheduler.state_dict())
        
# Updated test loop to calculate AUC
def test_model(model, test_loader, config, seed):
    model.eval()
    test_loss = 0.0
    all_predictions = []
    all_targets = []

    # Load the best model saved by early stopping
    model.load_state_dict(torch.load(f"./models/{config['architecture']}-{config['predictor']}-bestF1-seed_{seed}_opt_{config['optimizer']}_lr_{config['lr']}_batch_{config['batch_size']}_weight{config['positive_weight_factor']}.pth"))

    with torch.no_grad():
        for landsat, bioclim, sentinel, targets, _ in test_loader:
            landsat = landsat.to(config['device'])
            bioclim = bioclim.to(config['device'])
            sentinel = sentinel.to(config['device'])
            targets = targets.to(config['device'])

            outputs = model(landsat, bioclim, sentinel)
            pos_weight = targets * config["positive_weight_factor"]
            criterion = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight)
            test_loss += criterion(outputs, targets).item()

            # Collect raw predictions (probabilities after sigmoid) for AUC calculation
            probabilities = torch.sigmoid(outputs).cpu().numpy()
            all_predictions.append(probabilities)
            all_targets.append(targets.cpu().numpy())

    # Average the test loss
    test_loss /= len(test_loader)

    # Concatenate all predictions and targets
    all_predictions = np.concatenate(all_predictions, axis=0)
    all_targets = np.concatenate(all_targets, axis=0)

    # Calculate AUC (average='macro' or 'micro' may vary depending on your needs)
    # try:
        
    zero_cols_targets = np.all(all_targets == 0, axis=0)
    ones_cols_targets = np.all(all_targets == 1, axis=0)
    zero_cols = zero_cols_targets | ones_cols_targets

    filtered_targets = all_targets[:][:, ~zero_cols]

    list_of_lists = np.asarray([arr.tolist() for arr in all_predictions])
    filtered_probas = list_of_lists[:][:, ~zero_cols]

    # Calculate and print AUC (Area Under ROC Curve)
    auc = roc_auc_score(filtered_targets, filtered_probas, average='macro')

    # except ValueError:
    #     auc = None  # AUC calculation requires at least one positive and one negative sample

    # Calculate Precision, Recall, F1 scores using thresholding on predictions
    top_k_indices = np.argsort(-all_predictions, axis=1)[:, :25]  # Top-25 predictions per sample
    thresholded_predictions = np.zeros_like(all_predictions)
    rows = np.arange(len(all_predictions))[:, None]
    thresholded_predictions[rows, top_k_indices] = 1

    precision, recall, f1, _ = precision_recall_fscore_support(all_targets, thresholded_predictions, average='samples')

    # Log the results to W&B
    wandb.log({
        "test/loss": test_loss,
        "test/sF1@25": f1,
        "test/sPrecision@25": precision,
        "test/sRecall@25": recall,
        "test/AUC": auc
    })

    print(f"Test Loss: {test_loss}, Test Precision: {precision}, Test Recall: {recall}, Test F1: {f1}, Test AuC: {auc}")
        
# Validation function
def validate(model, val_loader, config):
    model.eval()
    val_loss = 0.0
    all_predictions, all_targets = [], []

    with torch.no_grad():
        for landsat, bioclim, sentinel, targets, _ in val_loader:
            landsat = landsat.to(config['device'])
            bioclim = bioclim.to(config['device'])
            sentinel = sentinel.to(config['device'])
            targets = targets.to(config['device'])

            outputs = model(landsat, bioclim, sentinel)
            pos_weight = targets * config['positive_weight_factor']
            criterion = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight)
            val_loss += criterion(outputs, targets).item()
            
            predictions = torch.sigmoid(outputs).cpu().numpy()
            top_k_indices = np.argsort(-predictions, axis=1)[:, :25]
            thresholded_predictions = np.zeros_like(predictions)
            rows = np.arange(len(predictions))[:, None]
            thresholded_predictions[rows, top_k_indices] = 1

            all_predictions.append(thresholded_predictions)
            all_targets.append(targets.cpu().numpy())
    
    val_loss /= len(val_loader)
    all_predictions = np.concatenate(all_predictions, axis=0)
    all_targets = np.concatenate(all_targets, axis=0)
    precision, recall, f1, _ = precision_recall_fscore_support(all_targets, all_predictions, average='samples', zero_division="warn")
    print(f"Validation Loss: {val_loss}, F1: {f1}, P: {precision}, R: {recall}")
    
    return val_loss, f1, precision, recall

# Function to split the data into training and validation sets
def split_train_val(train_metadata, val_size=0.05, seed=42):
    train_subset, val_subset = train_test_split(train_metadata, test_size=val_size, random_state=seed)
    return train_subset, val_subset

def run_experiment(train_metadata, val_metadata, test_metadata, train_transform, test_transform, param_combinations):
    
    for seed in [1, 66, 123, 777, 999]:  # Random seed for different experiment runs
        
        # Iterate over hyperparameter combinations
        for params in param_combinations:
            config = {
                'architecture': 'MME-Concat',
                'predictor': 'Landsat+Bioclim+Sentinel',
                'batch_size': params['batch_size'],
                'lr': params['lr'],
                'num_epochs': params['num_epochs'],
                'positive_weight_factor': params['positive_weight_factor'],
                'optimizer': params['optimizer'],  # Add optimizer to the config
                'device': torch.device('cuda' if torch.cuda.is_available() else 'cpu'),
            }

            # Initialize Weights and Biases (W&B) for logging
            initialize_wandb(config, run_name=f"{config['architecture']}-{config['predictor']}-seed_{seed}_opt_{params['optimizer']}_lr_{params['lr']}_batch_{params['batch_size']}_weight{params['positive_weight_factor']}")
            
                        
            # Create the train and validation datasets
            train_dataset = CustomDataset(bioclim_data_path, landsat_data_path, sentinel_data_path, train_metadata, subset="train", transform=transform)
            val_dataset = CustomDataset(bioclim_data_path, landsat_data_path, sentinel_data_path, val_metadata, subset="val", transform=transform)

            # Create DataLoader instances for train and validation
            train_loader = DataLoader(train_dataset, batch_size=config['batch_size'], shuffle=True, num_workers=8)
            val_loader = DataLoader(val_dataset, batch_size=config['batch_size'], shuffle=False, num_workers=8)

            # Create the test dataset and loader only once (as it's fixed)
            test_dataset = CustomDataset(test_bioclim_data_path, test_landsat_data_path, test_sentinel_data_path, test_metadata, subset="test", transform=test_transform)
            test_loader = DataLoader(test_dataset, batch_size=config['batch_size'], shuffle=False, num_workers=8)

            # Initialize the model, optimizer, scheduler, and early stopping
            model = MultiModalEnsembleC(NUM_CLASSES).to(config['device'])
            early_stopping = EarlyStopping(patience=5, verbose=True, path=f"./models/{config['architecture']}-{config['predictor']}-bestF1-seed_{seed}_opt_{params['optimizer']}_lr_{params['lr']}_batch_{params['batch_size']}_weight{params['positive_weight_factor']}.pth")

            # Train and validate
            set_seed(seed)
            train_and_validate(model, train_loader, val_loader, config, early_stopping)

            test_model(model, test_loader, config, seed)
                
            print(f"Finished training for seed {seed}.")

In [11]:
import albumentations as A


# Dataset and DataLoader
# transform = transforms.Compose([
#     transforms.ToTensor(),
#     transforms.Normalize(mean=(0.5, 0.5, 0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5, 0.5, 0.5)),
# ])
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.5, 0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5, 0.5)),
])

transform_albumentations = A.Compose([
                A.RandomBrightnessContrast(p=0.2),
                A.ColorJitter(p=0.2),
                A.OpticalDistortion(p=0.2),
            ])

class CustomDataset(Dataset):
    def __init__(self, bioclim_data_dir, landsat_data_dir, sentinel_data_dir, metadata, subset, transform=None):
        self.subset = subset
        self.transform = transform
        self.bioclim_data_dir = bioclim_data_dir
        self.landsat_data_dir = landsat_data_dir
        self.sentinel_data_dir = sentinel_data_dir
        self.metadata = metadata
        self.metadata = self.metadata.dropna(subset="speciesId").reset_index(drop=True)
        self.metadata['speciesId'] = self.metadata['speciesId'].astype(int)
        self.label_dict = self.metadata.groupby('surveyId')['speciesId'].apply(list).to_dict()
        
        self.metadata = self.metadata.drop_duplicates(subset="surveyId").reset_index(drop=True)


    def __len__(self):
        return len(self.metadata)

    def __getitem__(self, idx):
        
        survey_id = self.metadata.surveyId[idx]
        
        if self.subset in ["train", "val"]:
            landsat_sample = torch.nan_to_num(torch.load(os.path.join(self.landsat_data_dir, f"GLC24-PA-train-landsat-time-series_{survey_id}_cube.pt")))
            bioclim_sample = torch.nan_to_num(torch.load(os.path.join(self.bioclim_data_dir, f"GLC24-PA-train-bioclimatic_monthly_{survey_id}_cube.pt")))

        else:
            landsat_sample = torch.nan_to_num(torch.load(os.path.join(self.landsat_data_dir, f"GLC24-PA-test-landsat_time_series_{survey_id}_cube.pt")))
            bioclim_sample = torch.nan_to_num(torch.load(os.path.join(self.bioclim_data_dir, f"GLC24-PA-test-bioclimatic_monthly_{survey_id}_cube.pt")))
        
        species_ids = self.label_dict.get(survey_id, [])  # Get list of species IDs for the survey ID
        label = torch.zeros(NUM_CLASSES)  # Initialize label tensor
        for species_id in species_ids:
            #label_id = self.species_mapping[species_id]  # Get consecutive integer label
            label_id = species_id
            label[label_id] = 1  # Set the corresponding class index to 1 for each species
        
        rgb_sample = np.array(Image.open(construct_patch_path(self.sentinel_data_dir, survey_id)))
        nir_sample = np.array(Image.open(construct_patch_path(self.sentinel_data_dir.replace("rgb", "nir"), survey_id)))
        # altitude_sample = np.array(Image.open(construct_patch_path("../../satelite/elevation/", survey_id)))
        
        rgb_sample = transform_albumentations(image=rgb_sample)
        nir_sample = transform_albumentations(image=nir_sample)
        # altitude_sample = transform_albumentations(image=altitude_sample)
        
        # sentinel_sample = np.concatenate((rgb_sample["image"], nir_sample["image"][...,None], altitude_sample[...,None]), axis=2)    
        sentinel_sample = np.concatenate((rgb_sample["image"], nir_sample["image"][...,None]), axis=2)    

        sentinel_sample = self.transform(sentinel_sample)


        return landsat_sample, bioclim_sample, sentinel_sample, label, survey_id

In [12]:
# Initialize Weights and Biases
def initialize_wandb(config, run_name="experiment", project_name="GeoPlant"):
    wandb.init(project=project_name, name=run_name, config=config)

In [ ]:
import torch.optim as optim
import itertools

NUM_CLASSES = 11255 # Number of unique classes

# File paths for training and testing data
landsat_data_path = "../../SatelliteTimeSeries/GLC24-PA-train-landsat-time-series/"
bioclim_data_path = "../../EnvironmentalRasters/Climate/GLC24-PA-train-bioclimatic_monthly/"
sentinel_data_path= "../../satelite/pa_train_patches_rgb/"

label_path_PA = "../../metadata/GLC24_PA_metadata_train.csv"
dev_metadata = pd.read_csv(label_path_PA)

val_surveys = dev_metadata.drop_duplicates(subset="surveyId").sample(frac=0.05, random_state=1).surveyId.unique()

val_metadata = dev_metadata[dev_metadata.surveyId.isin(val_surveys)].reset_index(drop=True)
train_metadata = dev_metadata[~dev_metadata.surveyId.isin(val_surveys)].reset_index(drop=True)

test_landsat_data_path = "../../SatelliteTimeSeries/GLC24-PA-test-landsat_time_series/"
test_bioclim_data_path = "../../EnvironmentalRasters/Climate/GLC24-PA-test-bioclimatic_monthly/"
test_sentinel_data_path = "../../satelite/pa_test_patches_rgb/"

test_label_path = "../../metadata/PA_test_anonymized_glc24_private.csv"
test_metadata = pd.read_csv(test_label_path, sep=";")
test_metadata = test_metadata[test_metadata.surveyId != 2971941]


param_grid = {
    'batch_size': [32],
    'lr': [0.00001],
    'num_epochs': [50],
    'positive_weight_factor': [1],
    'optimizer': ['AdamW']  # List of optimizers to use
}

# Function to generate all combinations of hyperparameters
def create_param_combinations(param_grid):
    keys, values = zip(*param_grid.items())
    return [dict(zip(keys, combination)) for combination in itertools.product(*values)]

# Generate all possible hyperparameter combinations
param_combinations = create_param_combinations(param_grid)

# Run the experimentnum_classes
run_experiment(train_metadata, val_metadata, test_metadata, train_transform=transform, test_transform=transform, param_combinations=param_combinations)

Failed to detect the name of this notebook, you can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: Currently logged in as: picekl. Use `wandb login --relogin` to force relogin


Epoch 1/50, Train Loss: 0.03336513547027973
Validation Loss: 0.006490868040626602, F1: 0.220076329632797, P: 0.18858170375365252, R: 0.31348560136237563
Validation loss decreased (inf --> 0.006491).  Saving model ...
Scheduler: {'factor': 0.9, 'min_lrs': [0], 'patience': 1, 'verbose': False, 'cooldown': 0, 'cooldown_counter': 0, 'mode': 'min', 'threshold': 0.0001, 'threshold_mode': 'rel', 'best': 0.006490868040626602, 'num_bad_epochs': 0, 'mode_worse': inf, 'eps': 1e-08, 'last_epoch': 1, '_last_lr': [1e-05]}
Epoch 2/50, Train Loss: 0.005781825792952458
Validation Loss: 0.005263720748813025, F1: 0.3140510211722697, P: 0.26722409530231517, R: 0.45483436771611063
Validation loss decreased (0.006491 --> 0.005264).  Saving model ...
Scheduler: {'factor': 0.9, 'min_lrs': [0], 'patience': 1, 'verbose': False, 'cooldown': 0, 'cooldown_counter': 0, 'mode': 'min', 'threshold': 0.0001, 'threshold_mode': 'rel', 'best': 0.005263720748813025, 'num_bad_epochs': 0, 'mode_worse': inf, 'eps': 1e-08, 'la

test/AUC,▁
test/loss,▁
test/sF1@25,▁
test/sPrecision@25,▁
test/sRecall@25,▁
train/loss,█▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/lr,███████████████▆▆▆▆▆▆▆▆▆▆▄▄▄▄▄▄▄▄▄▄▄▂▂▁
val/loss,█▅▃▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val/sF1@25,▁▄▆▆▇▇▇▇▇▇█████████████████████████████
val/sPrecision@25,▁▄▆▆▇▇▇▇▇▇█████████████████████████████
val/sRecall@25,▁▄▆▆▇▇▇▇▇▇█████████████████████████████


Epoch 1/50, Train Loss: 0.03304200321837786
Validation Loss: 0.006488625751808285, F1: 0.224351383673747, P: 0.1918453585075298, R: 0.321274192264337
Validation loss decreased (inf --> 0.006489).  Saving model ...
Scheduler: {'factor': 0.9, 'min_lrs': [0], 'patience': 1, 'verbose': False, 'cooldown': 0, 'cooldown_counter': 0, 'mode': 'min', 'threshold': 0.0001, 'threshold_mode': 'rel', 'best': 0.006488625751808285, 'num_bad_epochs': 0, 'mode_worse': inf, 'eps': 1e-08, 'last_epoch': 1, '_last_lr': [1e-05]}
Epoch 2/50, Train Loss: 0.005795144098021236
Validation Loss: 0.005262691339677466, F1: 0.31305351982869484, P: 0.26681051921780174, R: 0.45208426327902684
Validation loss decreased (0.006489 --> 0.005263).  Saving model ...
Scheduler: {'factor': 0.9, 'min_lrs': [0], 'patience': 1, 'verbose': False, 'cooldown': 0, 'cooldown_counter': 0, 'mode': 'min', 'threshold': 0.0001, 'threshold_mode': 'rel', 'best': 0.005262691339677466, 'num_bad_epochs': 0, 'mode_worse': inf, 'eps': 1e-08, 'last

In [ ]:
wandb.finish()